[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/26_lora.ipynb)

# 🟠 Medium: LoRA (Low-Rank Adaptation)

Реализуйте **LoRA** — parameter-efficient fine-tuning линейного слоя. Вместо изменения большой frozen-матрицы `W₀` обучается low-rank поправка `ΔW = BA`, где rank `r` намного меньше входной и выходной dimension.

$$h = W_0 x + \frac{\alpha}{r} B A x$$

### Почему параметров меньше
Полная матрица содержит `out_features × in_features` параметров. LoRA хранит `A` формы `(r,in)` и `B` формы `(out,r)`, то есть `r(in+out)` обучаемых чисел. При малом `r` экономятся память optimizer state и gradient communication.

Для batch-first PyTorch tensors та же формула записывается справа налево: сначала `x @ A.T`, затем результат `@ B.T`. Base output и LoRA update имеют одинаковую последнюю dimension `out_features` и складываются.

### Инициализация и scaling
`A` инициализируется небольшими случайными значениями, `B` — нулями. Поэтому в момент создания `ΔW=0` и слой точно воспроизводит frozen base. После первых updates `B` становится ненулевой, и обе low-rank matrices начинают влиять на результат. Множитель `alpha/r` отделяет масштаб update от выбранного rank.

### Контракт
```python
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Требования
- `self.linear`: frozen `nn.Linear`, weight и bias имеют `requires_grad=False`;
- `self.lora_A`: `nn.Parameter(rank, in_features)`, random initialization;
- `self.lora_B`: `nn.Parameter(out_features, rank)`, **zero initialization**;
- scaling: `alpha / rank`.

### План реализации
1. Создайте base Linear и заморозьте каждый его parameter.
2. Создайте A и B с указанными shapes и initialization.
3. В forward отдельно вычислите base output и low-rank path.
4. Масштабируйте update и прибавьте к base output.

### Быстрые самопроверки
- сразу после initialization output равен `self.linear(x)`;
- shapes A и B совпадают с rank decomposition;
- base parameters не получают gradients;
- A и B зарегистрированы как parameters и получают `.grad`;
- после ненулевой B forward совпадает с явным `x @ A.T @ B.T`.

### Типичные ошибки
- неверный порядок A и B или transpose;
- base bias остался trainable;
- обе matrices инициализированы нулями: learning не стартует;
- забытый коэффициент `alpha/r`;
- замена base output вместо добавления update.

### Официальные материалы и примеры
- [torchtune LoRA overview](https://docs.pytorch.org/torchtune/stable/overview.html) — PyTorch-native PEFT recipes и ссылки на LoRA tutorial;
- [LoRA-enabled Gemma builder](https://docs.pytorch.org/torchtune/stable/generated/torchtune.models.gemma.lora_gemma_7b.html) — реальные параметры rank, alpha и выбор адаптируемых projections;
- [Оригинальная статья LoRA](https://arxiv.org/abs/2106.09685) — мотивация и low-rank formulation.

### Вопросы для самопроверки
1. Почему zero initialization B сохраняет исходное поведение слоя?
2. Сколько trainable parameters даёт LoRA по сравнению с полной matrix?
3. Зачем base parameters должны быть frozen?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0):
        super().__init__()
        pass  # frozen linear + lora_A + lora_B

    def forward(self, x):
        pass  # base + lora

In [ ]:
# 🧪 Debug
layer = LoRALinear(16, 8, rank=4)
x = torch.randn(2, 16)
print('Output:', layer(x).shape)
print('Trainable:', sum(p.numel() for p in layer.parameters() if p.requires_grad))
print('Total:    ', sum(p.numel() for p in layer.parameters()))

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('lora')